In [1]:
import sys, os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from black_scholes import bs_call
from monte_carlo import mc_price

os.makedirs('../figures', exist_ok=True)

S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
bs_ref = bs_call(S, K, T, r, sigma)
sns.set_theme(style='darkgrid')
print(f'Setup OK — BS référence : {bs_ref:.6f}')


Setup OK — BS référence : 10.450584


## Validation Monte Carlo — Convergence vers Black-Scholes

La méthode Monte Carlo approxime l'espérance du payoff actualisé par la moyenne empirique
sur N trajectoires GBM. L'erreur standard décroît en **1/√N** (théorème central limite) :
plus on simule, plus on converge vers le prix analytique BS.

Ce notebook mesure cette convergence en traçant l'erreur absolue |MC − BS| en fonction
de N sur une échelle log-log. La pente théorique −1/2 confirme le régime MC.


In [2]:
n_sims_list = [100, 500, 1_000, 5_000, 10_000, 50_000, 100_000, 500_000]

prices, stderrs, errors = [], [], []
for n in n_sims_list:
    p, se = mc_price(S, K, T, r, sigma, n, 'call', seed=42)
    prices.append(p)
    stderrs.append(se)
    errors.append(abs(p - bs_ref))
    print(f'n={n:>8,}  prix={p:.5f}  stderr={se:.5f}  |err|={errors[-1]:.5f}')

prices  = np.array(prices)
stderrs = np.array(stderrs)
errors  = np.array(errors)


n=     100  prix=7.82577  stderr=1.01196  |err|=2.62482
n=     500  prix=9.86836  stderr=0.63587  |err|=0.58222
n=   1,000  prix=9.91198  stderr=0.44835  |err|=0.53860
n=   5,000  prix=10.16892  stderr=0.20586  |err|=0.28166
n=  10,000  prix=10.34518  stderr=0.14766  |err|=0.10540
n=  50,000  prix=10.45769  stderr=0.06614  |err|=0.00711
n= 100,000  prix=10.42054  stderr=0.04677  |err|=0.03004
n= 500,000  prix=10.45005  stderr=0.02086  |err|=0.00053


## Graphe log-log : erreur absolue vs N

En échelle log-log, une convergence en 1/√N apparaît comme une droite de **pente −0.5**.
La droite pointillée rouge est la pente théorique calibrée sur le premier point mesuré.
Si les points suivent cette droite, le simulateur est correct.


In [3]:
n_arr = np.array(n_sims_list, dtype=float)

# Droite théorique 1/sqrt(N), calibrée sur le premier point
C = errors[0] * np.sqrt(n_arr[0])
theory = C / np.sqrt(n_arr)

fig, ax = plt.subplots(figsize=(9, 5))

ax.loglog(n_arr, errors,  'o-',  color='steelblue', lw=2,  label='Erreur absolue |MC − BS|')
ax.loglog(n_arr, stderrs, 's--', color='seagreen',  lw=1.5, label='Erreur standard (stderr)')
ax.loglog(n_arr, theory,  '--',  color='tomato',    lw=1.5, label='Pente théorique 1/√N')

ax.set_title('Convergence Monte Carlo vers Black-Scholes (Call ATM)', fontsize=13)
ax.set_xlabel('Nombre de simulations N')
ax.set_ylabel('Erreur absolue (€)')
ax.legend()

# Annotation de la pente empirique entre premier et dernier point
slope = np.polyfit(np.log(n_arr), np.log(errors), 1)[0]
ax.text(0.65, 0.85, f'Pente mesurée ≈ {slope:.3f}',
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle='round', fc='white', alpha=0.8))

plt.tight_layout()
plt.savefig('../figures/06_mc_convergence.png', dpi=150)
plt.close(fig)
print(f'Saved: figures/06_mc_convergence.png')
print(f'Pente log-log mesurée : {slope:.4f}  (théorique : -0.5000)')


Saved: figures/06_mc_convergence.png
Pente log-log mesurée : -0.9082  (théorique : -0.5000)


## Tableau récapitulatif

Prix MC, erreur standard et IC 95 % pour chaque N.


In [4]:
import pandas as pd

df = pd.DataFrame({
    'N sims':         n_sims_list,
    'Prix MC':        prices,
    'Stderr':         stderrs,
    'IC 95% (±)':     1.96 * stderrs,
    '|Err vs BS|':    errors,
    'Err rel (%)':    errors / bs_ref * 100,
})
df['N sims'] = df['N sims'].apply(lambda x: f'{x:,}')
print(df.to_string(index=False, float_format=lambda x: f'{x:.5f}'))
print(f'\nBS référence : {bs_ref:.6f}')


 N sims  Prix MC  Stderr  IC 95% (±)  |Err vs BS|  Err rel (%)
    100  7.82577 1.01196     1.98345      2.62482     25.11645
    500  9.86836 0.63587     1.24631      0.58222      5.57118
  1,000  9.91198 0.44835     0.87876      0.53860      5.15382
  5,000 10.16892 0.20586     0.40349      0.28166      2.69520
 10,000 10.34518 0.14766     0.28942      0.10540      1.00857
 50,000 10.45769 0.06614     0.12964      0.00711      0.06802
100,000 10.42054 0.04677     0.09167      0.03004      0.28747
500,000 10.45005 0.02086     0.04088      0.00053      0.00511

BS référence : 10.450584
